# Movie Recommendation System - EDA

This notebook is the exploratory analysis workspace for the Syntecxhub Movie Recommendation System project.

## Goals
- Load and inspect the raw MovieLens dataset
- Understand movie, rating, tag, and link coverage
- Check missing values and data quality
- Inspect genre and rating distributions
- Preview the processed metadata table
- Run a small end-to-end recommendation sanity check


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.helpers import configure_logging, dataframe_overview
from src.data.load_data import load_movielens_dataset, build_dataset_summary
from src.data.preprocess import preprocess_movielens_metadata
from src.features.build_features import build_feature_artifacts
from src.recommender.recommend import (
    build_runtime_assets_from_feature_artifacts,
    get_similar_movies_by_title,
    recommendation_results_to_dataframe,
)

configure_logging()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

PROJECT_ROOT


In [ ]:
dataset = load_movielens_dataset(data_dir=PROJECT_ROOT / "data" / "raw")
dataset_summary = build_dataset_summary(dataset)
dataset_summary

In [ ]:
print("Movies")
display(dataset.movies.head())

print("Ratings")
display(dataset.ratings.head())

print("Tags")
display(dataset.tags.head())

print("Links")
display(dataset.links.head())

In [ ]:
raw_overviews = {
    "movies": dataframe_overview(dataset.movies, "movies"),
    "ratings": dataframe_overview(dataset.ratings, "ratings"),
    "tags": dataframe_overview(dataset.tags, "tags"),
    "links": dataframe_overview(dataset.links, "links"),
}
pd.DataFrame(raw_overviews).T

In [ ]:
missing_values = pd.DataFrame({
    "movies_missing": dataset.movies.isna().sum(),
    "ratings_missing": dataset.ratings.isna().sum(),
    "tags_missing": dataset.tags.isna().sum(),
    "links_missing": dataset.links.isna().sum(),
}).fillna(0).astype(int)

missing_values

In [ ]:
ratings_stats = dataset.ratings["rating"].describe()
ratings_stats

In [ ]:
dataset.ratings["rating"].hist(bins=10)
plt.title("Ratings Distribution")
plt.xlabel("Rating")
plt.ylabel("Frequency")
plt.show()

In [ ]:
genre_counts = (
    dataset.movies.assign(
        genre=dataset.movies["genres"].fillna("").str.split("|")
    )
    .explode("genre")
)

genre_counts = genre_counts[
    genre_counts["genre"].notna()
    & (genre_counts["genre"] != "")
    & (genre_counts["genre"] != "(no genres listed)")
]

genre_counts["genre"].value_counts().head(15)

In [ ]:
processed_movies = preprocess_movielens_metadata(dataset)
processed_movies.head()

In [ ]:
processed_movies[[
    "movieId",
    "title",
    "clean_title",
    "release_year",
    "genres_text",
    "tags_text",
    "rating_count",
    "rating_mean",
    "metadata_text",
]].head(10)

In [ ]:
artifacts = build_feature_artifacts(processed_movies)
runtime_assets = build_runtime_assets_from_feature_artifacts(artifacts)

sample_recommendations = get_similar_movies_by_title(
    assets=runtime_assets,
    title="Toy Story",
    top_n=10,
    exclude_input_movie=True,
    min_rating_count=10,
)

recommendation_results_to_dataframe(sample_recommendations)